In [ ]:
include("MiniCollectiveSpins.jl")
using PyPlot
using Statistics
using JLD2
using ProgressBars

In [ ]:
function compute_Ω(I, γ, ω)
    return (6π*ustrip(c_0)^2*γ .* I / (ω^3 * ustrip(ħ))).^(1/2)
end

function decay_rate_indep(N, γ, Ω)
    return γ*N ./ (2 .+ (γ ./ Ω).^2)
end

### Define the system

In [ ]:
# Parameters
N = 10
# sat = vcat([0.1:0.1:0.9;], [1:35;])
sat = [40.1;]
sat_computed = sat#[1:4]
pathname_density_distrib = "Groundstate_Bext_90_deg_as_98.h5"
n_list = [10:50:1000;]  # List of densities to simulate.

# Quantization axis along z
e = [0, 0, 1.]

# Constants
λ = 421e-9
γ = 32.7e6 # In Hz
@load "op_list/op_list_$N.jdl2" op_list;

In [ ]:
println("Nbr of Atoms simulated = $(length(n_list)*N)")

### Recover data from simulations

In [ ]:
list_t_N, popup_t_N, sol_t_N, nbr_error_N = [], [], [], []

for (i, s) in ProgressBar(enumerate(sat_computed))
    line = []
    @load "solutions/Sat_$(minimum(sat))to$(maximum(sat))_n0_$(round(mean(n_list), digits=2))_N_$N/sol_N_$(N)_Sat_$(sat[i])_stripe_distribution_$(pathname_density_distrib[1:end-3])_n0_$(round(mean(n_list), digits=2)).jld2" N n_list sat sol_tasks
    list_t, popup_t, sol_t =  [vcat([st[i] for st in sol_tasks]...) for i = 1:3]
    push!(list_t_N, list_t), push!(popup_t_N, popup_t), push!(sol_t_N, sol_t), push!(nbr_error_N, sum(vcat([st[4] for st in sol_tasks]...)))
end


# Plots of the time evolution

### Density dependancy

In [ ]:
close("all")
fig, ax = subplots()

i = 1 # Saturation intensity

rc("font", size=12)

line = []
for j in 1:5:length(popup_t_N[i])
    line, = ax.plot(list_t_N[i][j], popup_t_N[i][j] / N, label="n0=$(round(Int, n_list[j]))")
end


ax.set_xlabel(L"$\gamma t$", fontsize=12)
ax.set_ylabel(L"$\langle  n_{\uparrow} \rangle $", fontsize=12)
ax.legend()

suptitle(L"$s=$"*"$(sat[i])")
pygui(false); gcf();
# pygui(true); show()
# savefig("Time_evol.pdf")

# Compute $\gamma_{SE}$ from the SR SS

In [ ]:
I_SE_SR_SS = zeros(ComplexF64, (length(sat_computed), length(n_list)))

for (i, s) in enumerate(sat_computed)
    for j in 1:length(n_list)
        
        if length(sol_t_N[i][j][end]) > 1 # If the full solution yas saved
            sol_SS = sol_t_N[i][j][end]
        else
            sol_SS = sol_t_N[i][j] # Only the SS was saved
        end

        for a = 1:N
            I_SE_SR_SS[i, j] += sol_SS[a] # The decay rates are normalized by γ
        end
    end
end
I_SE_SR_SS = real.(I_SE_SR_SS); # This is the mean decay rate of SE in the full stripe

# Compute $\gamma_{SR}$ from the SR SS

In [ ]:
I_SR_SR_SS = zeros(ComplexF64, (length(sat_computed), length(n_list)))

for (i, s) in ProgressBar(enumerate(sat_computed))
    for j in 1:length(n_list)
        @load "r0/Sat_$(minimum(sat))to$(maximum(sat))_n0_$(round(mean(n_list), digits=2))_N_$N/r0_N_$(N)_density_idx_$(i)_sat_$(round(sat[i], digits=1))_n0_$(round(mean(n_list), digits=2)).jdl2" r0 L

        system = SpinCollection(r0, e, gammas=1.)
        Γ_CS = GammaMatrix(system)
        
        if length(sol_t_N[i][j][end]) > 1 # If the full solution yas saved
            sol_SS = sol_t_N[i][j][end]
        else
            sol_SS = sol_t_N[i][j] # Only the SS was saved
        end

        for a = 1:N
            for b = 1:N
                if b > a # Correlated decay
                    corr = [21*10^(floor(Int, log10(a))+1)+a, 12*10^(floor(Int, log10(b))+1)+b]
                    try
                        I_SR_SR_SS[i, j] += 2*Γ_CS[a, b]*sol_SS[findall(x->x==corr || x==reverse(corr), op_list)[1]]
                    catch
                        println("Error @ $N, r=$j, a=$a, b=$b, corr=$corr")
                    end
                end
            end
        end
    end
end
I_SR_SR_SS = real.(I_SR_SR_SS); # This is the mean decay rate of SR in the full stripe

In [ ]:
Itot = I_SE_SR_SS .+ I_SR_SR_SS;

In [ ]:
close("all")
fig = subplots()

plot(n_list, reshape(I_SE_SR_SS, length(I_SE_SR_SS)), label="SE from SR SS")
plot(n_list, reshape(I_SR_SR_SS, length(I_SR_SR_SS)), label="Collective decay from SR SS")
plot(n_list, reshape(Itot, length(Itot)), label="Sum of the decays")

xlabel(L"s")
ylabel(L"$I$")

suptitle("Total nbr of atoms decaying, Starting from "*L"$|\downarrow \downarrow \rangle $, stde")
legend()

# pygui(true); show()
pygui(false);